# The pure agent baseline on Pendulum

This notebook follows the recipe in [*Now what? A recipe for after the problem
setting*](https://tomsilver.github.io/blog/2026/now-whats-your-solution/) on a
problem small enough to run end to end: swinging up an inverted pendulum.

1. **A concrete problem.** Gymnasium's `Pendulum-v1`.
2. **The stupidest approach.** Uniformly random torques.
3. **The pure agent.** Every abstract method is implemented by handing its
   inputs to a coding agent, using `prpl-agent-utils`.
4. **An oracle.** Energy-shaping swing-up with a PD catch, written with full
   knowledge of the dynamics.

Running the pure agent costs real API budget (a few tens of cents at the
defaults below) and takes several minutes.

In [1]:
import abc
import importlib.util
import sys
from pathlib import Path

import gymnasium as gym
import numpy as np

from prpl_agent_utils import ClaudeCodeAgent

ENV_ID = "Pendulum-v1"
MAX_STEPS = 200
TRAIN_SEEDS = list(range(5))
EVAL_SEEDS = list(range(100, 120))

# The agent runs in a Docker container by default; build the image once with
# `bash docker/build.sh`. Set to False to run the CLI directly on the host.
USE_DOCKER = True
SANDBOX_DIR = Path("pendulum_sandbox")
NUM_AGENT_ROUNDS = 3

## Step 1: a concrete problem

`Pendulum-v1` gives an observation of $(\cos\theta, \sin\theta,
\dot\theta)$ with $\theta = 0$ upright, takes a torque in $[-2, 2]$, and pays
a cost $-(\theta^2 + 0.1\dot\theta^2 + 0.001u^2)$ at every one of 200 steps.
The pendulum starts near the bottom, and the torque limit is too small to lift
it directly, so a solution has to pump energy in over several swings.

The interface is the part that matters for the recipe. A `Method` learns from
training problems, prepares for a new problem, and produces an action for each
observation.

In [2]:
class Method(abc.ABC):
    """A method for solving a sequential decision-making problem."""

    @abc.abstractmethod
    def train(self, train_seeds: list[int]) -> None:
        """Learn whatever you want from the training problems."""

    @abc.abstractmethod
    def reset(self, obs: np.ndarray, info: dict) -> None:
        """Prepare to solve a new problem."""

    @abc.abstractmethod
    def step(self, obs: np.ndarray) -> np.ndarray:
        """Given the current observation, return the next action."""


def evaluate(method: Method, seeds: list[int]) -> tuple[float, float]:
    """Return the mean and standard deviation of returns over the seeds."""
    env = gym.make(ENV_ID)
    returns = []
    for seed in seeds:
        obs, info = env.reset(seed=seed)
        method.reset(obs, info)
        total = 0.0
        for _ in range(MAX_STEPS):
            obs, reward, terminated, truncated, info = env.step(method.step(obs))
            total += float(reward)
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return float(np.mean(returns)), float(np.std(returns))

## Step 2: the stupidest possible approach

Random torques. This is here to check that the metric separates good behavior
from bad: if random actions scored well, the problem or the metric would be
wrong.

In [3]:
class RandomMethod(Method):
    """Sample a uniformly random torque at every step."""

    def __init__(self, seed: int = 0) -> None:
        self._action_space = gym.make(ENV_ID).action_space
        self._action_space.seed(seed)

    def train(self, train_seeds: list[int]) -> None:
        pass

    def reset(self, obs: np.ndarray, info: dict) -> None:
        pass

    def step(self, obs: np.ndarray) -> np.ndarray:
        return self._action_space.sample()


random_result = evaluate(RandomMethod(), EVAL_SEEDS)
print(f"random: {random_result[0]:.1f} +/- {random_result[1]:.1f}")

random: -1354.5 +/- 298.9


## Step 2.5: the pure agent

The blog's version of this step queries an agent inside `step`:

```python
def step(self, obs: Observation) -> Action:
    prompt = PROMPT_TEMPLATE.format(...)
    response = query_agent(prompt)
    return parse_action(response)
```

That is not workable here, and the reason is worth stating plainly: `Pendulum-v1`
runs at 20 Hz, so a single episode is 200 actions and an evaluation is 4000. No
agent closes that loop at the necessary rate or price. This is one of the
limitations the blog points to when it argues where research remains
competitive, and a pendulum is enough to run into it.

So the pure agent writes the policy instead of being the policy. During `train`,
it authors `policy.py` in its sandbox; we evaluate that file on the training
seeds and hand the score back; it revises. `step` then calls the function it
wrote. The agent's sandbox and its conversation both persist across the calls to
`query`, so each round builds on the last.

The prompt names the environment, which is a large hint: the agent has certainly
read about pendulum swing-up. That is the honest version of this baseline, and
the blog's warning applies — if it does well, the problem may be too easy rather
than the method too good.

In [4]:
INITIAL_PROMPT = """\
Write a file `policy.py` in the current directory containing a control policy \
for the gymnasium environment {env_id}.

The file must define exactly this function:

    def policy(obs):
        # obs is a numpy array [cos(theta), sin(theta), theta_dot],
        # with theta = 0 upright.
        # Return a numpy array of shape (1,) holding a torque in [-2, 2].

Only `numpy` may be imported. The function must be deterministic and fast: it is \
called 200 times per episode. Do not run anything; the policy is evaluated \
outside this sandbox and you will be told the score.

The score is the mean return over {num_seeds} episodes of {max_steps} steps, \
where each step pays -(theta^2 + 0.1*theta_dot^2 + 0.001*torque^2). Random \
torques score about -1200. Higher is better.
"""

FEEDBACK_PROMPT = """\
Your policy scored a mean return of {score:.1f} over the training episodes.

Revise `policy.py` to score higher. Keep the same function signature.
"""

ERROR_PROMPT = """\
Your policy could not be evaluated: {error}

Fix `policy.py`. Keep the same function signature.
"""


class PureAgentMethod(Method):
    """A method whose policy is written by a coding agent in its sandbox."""

    def __init__(
        self,
        sandbox_dir: Path,
        num_rounds: int = 3,
        use_docker: bool = True,
    ) -> None:
        self._agent = ClaudeCodeAgent(
            sandbox_dir,
            use_docker=use_docker,
            max_budget_usd_per_query=1.0,
        )
        self._num_rounds = num_rounds
        self._policy_fn = None
        self.history: list[tuple[int, float | None, str | None]] = []

    def train(self, train_seeds: list[int]) -> None:
        prompt = INITIAL_PROMPT.format(
            env_id=ENV_ID, num_seeds=len(train_seeds), max_steps=MAX_STEPS
        )
        for round_idx in range(self._num_rounds):
            self._agent.query(prompt)
            policy_fn, error = self._load_policy()
            score: float | None = None
            if error is None:
                assert policy_fn is not None
                try:
                    score = evaluate(_FixedPolicyMethod(policy_fn), train_seeds)[0]
                    self._policy_fn = policy_fn
                except Exception as exc:  # the policy ran but misbehaved
                    error = f"{type(exc).__name__}: {exc}"
            self.history.append((round_idx, score, error))
            print(f"round {round_idx}: score={score}, error={error}")
            if error is not None:
                prompt = ERROR_PROMPT.format(error=error)
            else:
                assert score is not None
                prompt = FEEDBACK_PROMPT.format(score=score)

    def _load_policy(self):
        """Import `policy.py` from the sandbox; return (function, error)."""
        path = self._agent.sandbox_dir / "policy.py"
        if not path.exists():
            return None, "policy.py was not created"
        try:
            spec = importlib.util.spec_from_file_location("agent_policy", path)
            assert spec is not None and spec.loader is not None
            module = importlib.util.module_from_spec(spec)
            sys.modules["agent_policy"] = module
            spec.loader.exec_module(module)
            policy_fn = module.policy
            probe = np.array([1.0, 0.0, 0.0])
            action = np.asarray(policy_fn(probe), dtype=np.float64).reshape(1)
            assert np.isfinite(action).all()
        except Exception as exc:
            return None, f"{type(exc).__name__}: {exc}"
        return policy_fn, None

    def reset(self, obs: np.ndarray, info: dict) -> None:
        pass

    def step(self, obs: np.ndarray) -> np.ndarray:
        assert self._policy_fn is not None, "train() must produce a policy first"
        action = np.asarray(self._policy_fn(obs), dtype=np.float64).reshape(1)
        return np.clip(action, -2.0, 2.0)


class _FixedPolicyMethod(Method):
    """Wrap a plain policy function so it can be evaluated as a Method."""

    def __init__(self, policy_fn) -> None:
        self._policy_fn = policy_fn

    def train(self, train_seeds: list[int]) -> None:
        pass

    def reset(self, obs: np.ndarray, info: dict) -> None:
        pass

    def step(self, obs: np.ndarray) -> np.ndarray:
        action = np.asarray(self._policy_fn(obs), dtype=np.float64).reshape(1)
        return np.clip(action, -2.0, 2.0)

The agent is sandboxed while it writes the policy, but loading `policy.py` here
runs its code in this notebook's process. The sandbox protects the authoring
step, not the evaluation step. Evaluating inside the container too, and passing
only the score back, would close that gap.

In [5]:
agent_method = PureAgentMethod(
    SANDBOX_DIR, num_rounds=NUM_AGENT_ROUNDS, use_docker=USE_DOCKER
)
agent_method.train(TRAIN_SEEDS)
agent_result = evaluate(agent_method, EVAL_SEEDS)
print(f"pure agent: {agent_result[0]:.1f} +/- {agent_result[1]:.1f}")

round 0: score=-143.36494786121958, error=None


round 1: score=-142.73398565730787, error=None


round 2: score=-142.60222849514494, error=None
pure agent: -184.2 +/- 75.5


In [6]:
print((SANDBOX_DIR / "policy.py").read_text())

import numpy as np

# Pendulum-v1 physical constants (see gym's pendulum.py dynamics):
#   theta_ddot = (3*g / (2*l)) * sin(theta) + (3 / (m*l**2)) * u
# with theta = 0 at the (unstable) upright equilibrium.
_G = 10.0
_L = 1.0
_M = 1.0
_MAX_TORQUE = 2.0
_K = 3.0 * _G / (2.0 * _L)   # = 15.0, gravity coefficient
_B = 3.0 / (_M * _L ** 2)    # = 3.0,  torque coefficient
_E_TOP = _K                  # conserved energy 0.5*thetadot^2 + K*cos(theta) at rest, upright

# Switch to a linear stabilizer once close enough to upright to hold statically
# (max torque can balance gravity there: K*theta < B*MAX_TORQUE for theta < 0.4).
_ANGLE_THRESH = 0.3
_SPEED_THRESH = 2.0

# LQR gains for the linearized balance dynamics theta_ddot = K*theta + B*u,
# solved for the *actual* per-step cost weights Q = diag(1, 0.1), R = 0.001
# (continuous-time ARE, solved in closed form), so the stabilizer directly
# minimizes theta^2 + 0.1*theta_dot^2 + 0.001*u^2 near the top rather than
# using an arbitrarily chose

## Step 3: an oracle

The oracle uses privileged knowledge: the exact dynamics gymnasium implements,
$\ddot\theta = \frac{3g}{2l}\sin\theta + \frac{3}{ml^2}u$. Pumping energy
toward the value it has standing upright, then catching it with a PD controller,
is the textbook solution. This establishes what a good score looks like.

In [7]:
G, L, M = 10.0, 1.0, 1.0
E_UPRIGHT = 3 * G / (2 * L)


class OracleMethod(Method):
    """Energy-shaping swing-up with a PD controller near the top."""

    def __init__(self, gain: float = 0.6, kp: float = 20.0, kd: float = 4.0,
                 switch: float = 0.6) -> None:
        self._gain, self._kp, self._kd, self._switch = gain, kp, kd, switch

    def train(self, train_seeds: list[int]) -> None:
        pass

    def reset(self, obs: np.ndarray, info: dict) -> None:
        pass

    def step(self, obs: np.ndarray) -> np.ndarray:
        cos_th, sin_th, thdot = obs
        theta = np.arctan2(sin_th, cos_th)
        if abs(theta) < self._switch:
            torque = -self._kp * theta - self._kd * thdot
        else:
            energy = 0.5 * thdot**2 + E_UPRIGHT * cos_th
            direction = np.sign(thdot) if thdot != 0 else 1.0
            torque = self._gain * (E_UPRIGHT - energy) * direction
        return np.clip([torque], -2.0, 2.0)


oracle_result = evaluate(OracleMethod(), EVAL_SEEDS)
print(f"oracle: {oracle_result[0]:.1f} +/- {oracle_result[1]:.1f}")

oracle: -184.4 +/- 75.3


## Results

In [8]:
results = {
    "random": random_result,
    "pure agent": agent_result,
    "oracle": oracle_result,
}
print(f"{'method':<12} {'mean return':>12} {'std':>8}")
for name, (mean, std) in results.items():
    print(f"{name:<12} {mean:>12.1f} {std:>8.1f}")

method        mean return      std
random            -1354.5    298.9
pure agent         -184.2     75.5
oracle             -184.4     75.3


## What this tells you

The pure agent tied the oracle: -184.2 against -184.4, well inside the spread
across evaluation seeds. Reading its `policy.py` shows why. It identified the
environment's dynamics from the name, derived the energy-shaping swing-up, and
then went past the hand-written oracle by solving the algebraic Riccati equation
for the balance controller using the actual cost weights from the reward
function, where the oracle uses PD gains chosen by hand.

That is the outcome the blog warns about, and it is informative rather than
disappointing. On this problem there is no gap between the stupidest sophisticated
method and the best one, so nothing here would justify a new method. The useful
response is to return to Step 1: dynamics the agent cannot name, observations it
cannot interpret, or an interaction budget that rules out writing and testing
code offline. The pure agent baseline is what tells you which of those you need.

Two properties of `prpl-agent-utils` carried the method above. The sandbox
persists, so `policy.py` from one round is there to be revised in the next. The
conversation persists, so the agent sees the score for the code it just wrote
without being re-told the problem. Both come from constructing the agent once and
holding it as a member of the method.